In [1]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

In [2]:
!pip install -q langchain langchain-core langchain-community pydantic duckduckgo-search ddgs langchain-experimental requests langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 97.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [23]:
from langchain_core.tools import tool
import requests

In [24]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool=DuckDuckGoSearchRun()

In [25]:
@tool
def get_weather_data(city:str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=0ff47689039b9b22cad26f73e19f4eec&query={city}'

  response=requests.get(url)

  return response.json()

In [26]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
)

llm = ChatHuggingFace(llm=llm)

In [27]:
from langchain_classic import hub
from langchain_classic.agents import create_react_agent, AgentExecutor

In [28]:
# step-1: Pull the ReAct prompt from the Langchain Hub

from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True
)

In [29]:
# step-2: Create the ReAct agent manually with the pulled prompt
agent=create_react_agent(
    llm=llm,
    tools=[search_tool],
    prompt=prompt
)

In [30]:
# step-3: Wrap it with AgentExecuter
agent_executor=AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True
)

In [31]:
# invoke
response = agent_executor.invoke({"input": "Find the capital of Madhya Pradesh, then find it's current weather condition"})



> Entering new AgentExecutor chain...
I need to find the capital of Madhya Pradesh first, and then check its current weather condition.
Action: duckduckgo_search
Action Input: capital of Madhya Pradesh
ObservIts capital is Bhopal. Other major cities include Indore, Gwalior, Jabalpur, Chhindwara, and Sagar. Madhya Pradesh is the second largest Indian state by area and the fifth largest state by population with over 72 million residents. Madhya Pradesh, state of India. As its name implies—madhya means “central” and pradesh means “region” or “state”—it is situated in the heart of the country. The state has no coastline and no international frontier. It is bounded by the states of Uttar Pradesh to the northeast, Chhattisgarh to the southeast, Maharashtra to the south, Gujarat to the ... See full list on britannica.com Madhya Pradesh lies over a transitional area between the Indo-Gangetic Plain in the north and the Deccan plateau in the south. Its physiography is characterized by low hill

In [32]:
response['output']

'The capital of Madhya Pradesh is Bhopal, and as of the last update, the current temperature in Bhopal is 29°C with a feel-like temperature of 32°C. The forecast for today shows a high of 41°C and a low of 29°C.'